In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# import os
# for dir, _, files in os.walk('kaggle/input'):


In [ ]:
!pip install lightning
# !pip install scikit-learn
# !pip install transformers

# Import constants in param.py

In [ ]:
# import os
import platform
from typing import Union

OS_PLATFORM = platform.system()

TRAIN_PATH = 'data/sp_data/train_set.fasta'
BENCHMARK_PATH = 'data/sp_data/benchmark_set_sp5.fasta'

SP_LABELS = dict(NO_SP=0, SP=1, LIPO=2, TAT=3, PILIN=4, TATLIPO=5)
# KINGDOM = dict(EUKARYA=0, POSITIVE=1, NEGATIVE=2, ARCHAEA=1)
KINGDOM = dict(POSITIVE=1, NEGATIVE=2, ARCHAEA=3)
"""
MODEL AND TRAINING CONFIGURATION
"""
ENV = 'local'
EPOCHS = 3 if OS_PLATFORM == 'Windows' else 50
BATCH_SIZE = 8
MODEL = 'cnn'
DATA = 'aa'
CONF_TYPE = 'default'
DEVICES: Union[list[int], str, int] = 'auto'
ACCELERATOR = 'auto'
NUM_WORKERS = 1 if OS_PLATFORM == 'Windows' else 2
ORGANISM = 'others'
FREEZE_PRETRAINED = False

DEVICE = 'cpu'  # use for apply old training process

"""
LOGGER CONFIGURATION
"""
INPUT_DIR = '/kaggle/input'
WORKING_DIR = '/kaggle/working'

LOG_DIR = 'logs'

# Directory _callbacks_

In [ ]:
# import all callback in callback_utils.py here
from pathlib import Path
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

model_checkpoint = ModelCheckpoint(
    dirpath=str(Path(WORKING_DIR, 'checkpoints')),
    filename=f"{MODEL}_{DATA}_epoch={EPOCHS}_{CONF_TYPE}_{ENV}",
    enable_version_counter=True,
    monitor='val_loss',
    every_n_epochs=1,
    save_on_train_epoch_end=True,
    save_top_k=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    min_delta=0.00,
    patience=3,
    verbose=False,
    mode="max"
)

print("Finished")

# Directory _configs_

In [ ]:
# import all utils in config_utils.py
import json
import os

from transformers import BertConfig


def load_config(model, data, conf_type):
    if model == "bert_pretrained":
        config = BertConfig().from_pretrained("Rostlab/prot_bert")
        return config
    elif model == "bert":
        with open(str(Path(INPUT_DIR, f'configs/{data}_configs/{model}_config_default.json'))) as f:
            data = json.load(f)
            config = BertConfig(**data)
            return config
    else:
        conf_path = str(Path(INPUT_DIR, f'configs/{data}_configs/{model}_config_{conf_type}.json'))
        if os.path.exists(conf_path):
            with open(conf_path, 'r') as f:
                config = json.load(f)
                return config
        else:
            raise FileNotFoundError("Config file does not exist")


# Directory _models_

In [ ]:
# Neural Network Layers

import math

from torch import nn


class OrganismEmbedding(nn.Module):
    """
    Parameters `num_orgs` and `e_dim` must be the same
    """

    def __init__(self, num_orgs: int = 4, e_dim: int = 4):
        super().__init__()
        self.num_orgs = num_orgs
        self.embedding_dim = e_dim
        oe = torch.zeros(num_orgs, e_dim)
        for i in range(num_orgs):
            oe[i, i] = 1
        # oe = oe.unsqueeze(0)
        # self.register_buffer('oe', oe)
        self.oe = nn.Embedding.from_pretrained(torch.FloatTensor(oe), freeze=False)

    def forward(self, x):
        return self.oe(x)


class InputEmbedding(nn.Module):
    def __init__(self, vocab_size: int = 100, d_model: int = 512):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.input_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model
        )

    def forward(self, x):
        x = self.input_embedding(x) * math.sqrt(self.d_model)
        return x


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int = 512, dropout: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        pe_x = self.pe[:, :x.size(1), :]
        x = x + pe_x
        return self.dropout(x)


class LinearPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 512, dropout: float = 0.1):
        super().__init__()
        self.pe = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        return self.pe(x)


class TransformerEncoder(nn.Module):
    def __init__(self, d_model: int = 512, nhead: int = 8, num_layers: int = 6):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):
        x = self.encoder(x)
        # use average [CLS] token with all other word tokens
        # x = torch.mean(x, dim=1)

        # use only [CLS] token
        x = x[:, 0, :]
        return x


class ConvolutionalEncoder(nn.Module):
    def __init__(
            self,
            embedding_dim: int = 512,
            dropout: float = 0.1,
            kernel_size: int = 3,
            stride: int = 1,
            padding: int = 0,
            n_base: int = 1024,
    ):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=embedding_dim, out_channels=n_base, kernel_size=kernel_size, stride=stride,
                      padding=padding),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(in_channels=n_base, out_channels=n_base * 4, kernel_size=kernel_size, stride=stride,
                      padding=padding),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(in_channels=n_base * 4, out_channels=n_base, kernel_size=kernel_size, stride=stride,
                      padding=padding),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv1d(in_channels=n_base, out_channels=embedding_dim, kernel_size=kernel_size, stride=stride,
                      padding=padding),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.dropout(x)
        return x


class LSTMEncoder(nn.Module):
    def __init__(
            self,
            embedding_dim: int = 512,
            hidden_size: int = 1024,
            n_layers: int = 4,
            dropout: float = 0.1,
            random_init: bool = False
    ):
        super().__init__()
        self.random_init = random_init
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=n_layers,
            dropout=dropout,
            batch_first=True,
            bidirectional=False
        )

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        return out, h_n, c_n


class StackedBiLSTMEncoder(nn.Module):
    def __init__(
            self,
            embedding_dim: int = 512,
            hidden_size: int = 1024,
            n_layers: int = 4,
            dropout: float = 0.1,
            random_init: bool = False
    ):
        super().__init__()
        self.random_init = random_init
        # Init state
        if random_init:
            (h_0, c_0) = self.__init_state(n_layers=n_layers, hidden_size=hidden_size)
            self.init_state = (h_0, c_0)

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            bidirectional=True,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

    def forward(self, x):
        if self.random_init:
            out, (h_n, c_n) = self.bilstm(x, (self.init_state[0].detach(), self.init_state[1].detach()))
            return out, h_n, c_n
        else:
            out, (h_n, c_n) = self.bilstm(x)
            return out, h_n, c_n

    @staticmethod
    def __init_state(n_layers: int = 4, hidden_size: int = 1024):
        h_0 = torch.zeros(n_layers * 2, BATCH_SIZE, hidden_size).requires_grad_(True)
        c_0 = torch.zeros(n_layers * 2, BATCH_SIZE, hidden_size).requires_grad_(True)
        return h_0, c_0


class ParallelBiLSTMEncoder(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        pass


class Classifier(nn.Module):
    def __init__(self, num_class: int, d_model: int = 512, d_ff: int = 2048):
        super().__init__()
        self.ff1 = nn.Linear(in_features=d_model, out_features=d_ff)
        self.ff2 = nn.Linear(in_features=d_ff, out_features=num_class)
        self.act1 = nn.ReLU()
        # self.act2 = nn.Softmax()

    def forward(self, x):
        x = self.act1(self.ff1(x))
        # x = self.act2(self.ff2(x))
        x = self.ff2(x)
        return x

In [ ]:
# models

from torch import nn

""" Transformer """


class TransformerClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.positional_encoding = PositionalEncoding(
            d_model=config['d_model'],
            dropout=config['dropout'],
            max_len=config['max_len']
        )
        self.encoder = TransformerEncoder(
            d_model=config['d_model'],
            nhead=config['nhead'],
            num_layers=config['num_layers']
        )
        self.classifier = Classifier(
            d_model=config['d_model'],
            num_class=len(SP_LABELS)
        )

    def forward(self, x):
        # print("Is Nan Input: ", x.isnan().any())
        x = self.input_embedding(x)
        x = self.positional_encoding(x)
        x = self.encoder(x)
        # print("Is Nan Embedding: ", x.isnan().any(), x[:2,:10])
        # print(x[0, :)
        x = self.classifier(x)
        return x


""" CNN """


class ConvolutionalClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.conv_encoder = ConvolutionalEncoder(
            embedding_dim=config['d_model'],
            kernel_size=config['kernel_size'],
            n_base=config['n_base']
        )
        self.flatten = nn.Flatten()
        self.classifier = Classifier(num_class=len(SP_LABELS), d_model=1024)
        # self.dropout = nn.Dropout(p=0.1)

    def forward(self, x):
        x = self.input_embedding(x)
        x = torch.transpose(x, 1, 2)
        x = self.conv_encoder(x)
        x = self.flatten(x)
        #         print(x.shape)
        x = self.classifier(x)
        return x


""" LSTM """


class LSTMClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.encoder = LSTMEncoder(
            embedding_dim=config['d_model'],
            hidden_size=config['hidden_size'],
            n_layers=config['n_layers'],
            dropout=config['dropout'],

        )
        self.classifier = Classifier(num_class=len(SP_LABELS), d_model=config['hidden_size'] * 2)

    def forward(self, x):
        x = self.input_embedding(x)
        x, h_n, c_n = self.encoder(x)
        x = self.classifier(x[:, -1, :])
        return x


""" Stacked Bi-LTSM """


class StackedBiLSTMClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.stacked_encoder = StackedBiLSTMEncoder(
            embedding_dim=config['d_model'],
            hidden_size=config['hidden_size'],
            n_layers=config['n_layers'],
            dropout=config['dropout'],

        )
        self.classifier = Classifier(num_class=len(SP_LABELS), d_model=config['hidden_size'] * 2)

    def forward(self, x):
        x = self.input_embedding(x)
        x, h_n, c_n = self.stacked_encoder(x)
        x = self.classifier(x[:, -1, :])
        return x


""" CNN+Transformer """


class CNNTransformerClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.conv_encoder = ConvolutionalEncoder(
            embedding_dim=config['d_model'],
            kernel_size=config['kernel_size']
        )
        self.positional_encoding = PositionalEncoding(
            d_model=config['d_model'],
            dropout=config['dropout'],
            max_len=config['max_len']
        )
        self.trans_encoder = TransformerEncoder(
            d_model=config['d_model'],
            nhead=config['nhead'],
            num_layers=config['num_layers']
        )
        self.classifier = Classifier(
            d_model=config['d_model'],
            num_class=len(SP_LABELS)
        )

    def forward(self, x):
        x = self.input_embedding(x)
        x = torch.transpose(x, 1, 2)
        x = self.conv_encoder(x)
        x = torch.transpose(x, 1, 2)
        x = self.positional_encoding(x)
        x = self.trans_encoder(x)
        x = self.classifier(x)
        return x


""" ProtBERT """


class ProtBertClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BertModel(config=config)
        # self.bert_encoder = get_peft_model(encoder, peft_config)
        if FREEZE_PRETRAINED:
            self.freeze_pretrained_layer()
        self.classifier = Classifier(num_class=len(SP_LABELS), d_model=config.hidden_size)

    def forward(self, x):
        x = self.bert(x)
        x = x.last_hidden_state[:, 0, :]
        x = self.classifier(x)
        return x

    def freeze_pretrained_layer(self):
        for param in self.bert.parameters():
            param.requires_grad = False



In [ ]:
# import all model utils in model_utils.py
from transformers import BertModel

# Define types of model
TRANSFORMER = 'transformer'
CNN = 'cnn'
LSTM = 'lstm'
STACKED_BILSTM = 'st_bilstm'
BERT = 'bert'
BERT_PRETRAINED = 'bert_pretrained'
CNN_TRANSFORMER = 'cnn_trans'


def load_model(model, data, conf_type):
    config = load_config(model, data, conf_type)
    if model == TRANSFORMER:
        return TransformerClassifier(config)
    elif model == CNN:
        return ConvolutionalClassifier(config)
    elif model == STACKED_BILSTM:
        return StackedBiLSTMClassifier(config)
    elif model == BERT or model == BERT_PRETRAINED:
        return BertModel(config)
    elif model == LSTM:
        return LSTMClassifier(config)
    elif model == CNN_TRANSFORMER:
        return CNNTransformerClassifier(config)
    else:
        return ValueError("Unknown model type")


# Directory _data_

In [ ]:
# code SPDataset (extends torch.utils.data.Dataset)

# from transformers import BertTokenizer
# from tokenizers import Tokenizer


from typing import Optional

from torch.utils.data import Dataset


class SPDataset(Dataset):
    def __init__(self, json_paths: Optional[list[str]], dtype: str):
        if json_paths is None or isinstance(json_paths, str):
            raise ValueError('provide path to dataset in list of str')
        df = pd.concat(pd.read_json(path) for path in json_paths)
        self.smiles = df['smiles'].tolist()
        self.aa_seq = df['aa_seq'].tolist()
        self.labels = df['label'].tolist()
        self.kingdoms = df['kingdom'].tolist()
        self.dtype = dtype
        # tokenizer_path = ut.abspath(self._config['tokenizer_path'])
        # tokenizer = GPT2TokenizerFast(tokenizer_file=tokenizer_path)
        # if tokenizer.pad_token is None:
        #     tokenizer.add_special_tokens({'pad_token': '[PAD]'})
        # self.tokenizer = tokenizer
        # if params.MODEL == 'bert_pretrained':
        #     self.tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert")

    # @property
    # def _config(self):
    #     with open(ut.abspath(f'configs/data_config/{self.dtype}_config.json')) as f:
    #         config = json.load(f)
    #         return config

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, index):
        kingdom = self.kingdoms[index]
        seq = self.aa_seq[index] if self.dtype == 'aa' else self.smiles[index]
        label = torch.zeros(len(SP_LABELS), dtype=torch.int64)
        label[SP_LABELS[self.labels[index]]] = 1
        return seq, label, kingdom

    # def __getitem__(self, index):
    #     kingdom = self.kingdoms[index]
    #     seq = self.smiles[index] if self.dtype == 'smiles' else self.aa_seq[index]
    #     label = torch.zeros(len(params.SP_LABELS), dtype=torch.int64)
    #     label[params.SP_LABELS[self.labels[index]]] = 1
    #     input_ids = self.tokenizer.encode(
    #         seq,
    #         max_length=self._config['max_len'],
    #         padding="max_length",
    #         truncation=True
    #     )
    #     return torch.tensor(input_ids, dtype=torch.int64), label, kingdom


print("Finished")

# Directory _tokenizer_

In [ ]:
from transformers import GPT2TokenizerFast, BertTokenizer


# import utils from tokenizer_utils.py
def load_tokenizer(model, data):
    tokenizer_path = str(Path(INPUT_DIR, 'sppredictor-tokenizer', f'tokenizer/tokenizer_{data}.json'))
    tokenizer = GPT2TokenizerFast(tokenizer_file=tokenizer_path)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    if model == 'bert_pretrained':
        tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert")
    return tokenizer

# Directory _lightning_module_

In [ ]:
# code Lightning Data Module
from typing import Optional, Literal

import lightning as L
from lightning.pytorch.utilities.types import EVAL_DATALOADERS, TRAIN_DATALOADERS
from torch.utils.data import DataLoader


# from data.sp_dataset import SPDataset


class SPDataModule(L.LightningDataModule):
    def __init__(self, dataset_type: Literal["aa", "smiles"] = "smiles"):
        super().__init__()
        self.test_set = None
        self.val_set = None
        self.train_set = None
        self.batch_size = BATCH_SIZE
        self.num_workers = NUM_WORKERS

    # def prepare_data(self) -> None:
    #     data_utils = DataUtils()
    #     data_utils.extract_raw_dataset_by_partition(raw_path=str(Path(ROOT_DIR) / configs.TRAIN_PATH))
    #     # data_utils.extract_raw_dataset_by_partition(raw_path=str(Path(ROOT_DIR) / configs.BENCHMARK_PATH),
    #     #                                             benchmark=True)

    def setup(self, stage: Optional[str] = None) -> None:
        if stage == "fit" or stage is None:
            # print(f"\nSetting up: Using {self.dataset_type}\n")
            train_path = [str(Path(INPUT_DIR, 'sppredictor-dataset', 'train_set_partition_0.json')),
                          str(Path(INPUT_DIR, 'sppredictor-dataset', 'train_set_partition_1.json'))]
            val_path = [str(Path(INPUT_DIR, 'sppredictor-dataset', 'test_set_partition_0.json')),
                        str(Path(INPUT_DIR, 'sppredictor-dataset', 'test_set_partition_0.json'))]
            self.train_set = SPDataset(json_paths=train_path, dtype=DATA)
            self.val_set = SPDataset(json_paths=val_path, dtype=DATA)
        elif stage == "test":
            test_path = [str(Path(INPUT_DIR, 'sppredictor-dataset', 'train_set_partition_2.json')),
                         str(Path(INPUT_DIR, 'sppredictor-dataset', 'test_set_partition_2.json'))]
            self.test_set = SPDataset(json_paths=test_path, dtype=DATA)

    def train_dataloader(self) -> TRAIN_DATALOADERS:
        return DataLoader(self.train_set, batch_size=self.batch_size, shuffle=True,
                          num_workers=self.num_workers, persistent_workers=True, pin_memory=True)

    def val_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(self.val_set, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, persistent_workers=True, pin_memory=True)

    def test_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(self.test_set, batch_size=self.batch_size, shuffle=False)


print("Finished")

In [ ]:
import os.path
from typing import Any

import lightning as L
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from torch import optim, Tensor
from torch.nn import CrossEntropyLoss, Softmax
from torch.optim import Optimizer
from torchmetrics import F1Score, MatthewsCorrCoef, AveragePrecision


# from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef
# from torcheval.metrics.functional import multiclass_auroc, multiclass_auprc, multiclass_f1_score


class SPModule(L.LightningModule):
    def __init__(self):
        super().__init__()
        loss_weight = torch.tensor([1, 1, 1, 1, 1, 1], dtype=torch.float)
        self.loss_fn = CrossEntropyLoss(weight=loss_weight)
        self.save_hyperparameters()
        # self.fabric = Fabric()

        # Load config (Remove if unnecessary)
        # self.config = cut.load_config()

        # Tokenizer
        self.tokenizer = load_tokenizer(MODEL, DATA)

        # Load models
        self.model_type = MODEL
        self.model = load_model(MODEL, DATA, CONF_TYPE)

        # Load metrics
        self.f1 = F1Score(task='multiclass', num_classes=len(SP_LABELS), average='micro')
        self.mcc = MatthewsCorrCoef(task='multiclass', num_classes=len(SP_LABELS))
        self.average_precision = AveragePrecision(task='multiclass', num_classes=len(SP_LABELS))

        # Outputs from training process
        self.validation_step_outputs_lb = []
        self.validation_step_outputs_pred = []
        self.best_val_loss = 1e6

        self.test_outputs_lb = []
        self.test_outputs_pred = []
        self.test_step_outputs = {}

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.model.parameters(), lr=1e-5, weight_decay=0.1)
        return optimizer

    def optimizer_zero_grad(self, epoch: int, batch_idx: int, optimizer: Optimizer):
        optimizer.zero_grad(set_to_none=True)

    def backward(self, loss: Tensor, *args: Any, **kwargs: Any):
        loss.backward()

    def tokenize_input(self, x):
        encoded = self.tokenizer.batch_encode_plus(
            x,
            # max_length=self.model.config['max_len'],
            truncation=True,
            padding=True
        )
        # print(len(encoded['input_ids'][0]))
        return torch.tensor(encoded['input_ids'], dtype=torch.int64, device=self.device)

    def base_step(self, batch, batch_idx):
        x, lb, kingdom = batch
        x = self.tokenize_input(x)
        pred = self.model(x)
        loss = self.loss_fn(pred.float(), lb.float())
        return x, lb, pred, loss, kingdom

    def training_step(self, batch, batch_idx):
        _, _, pred, loss, _ = self.base_step(batch, batch_idx)
        self.log('train_loss', loss, on_epoch=True, prog_bar=True, logger=True, batch_size=BATCH_SIZE)
        return loss

    def validation_step(self, batch, batch_idx):
        _, lb, pred, loss, _ = self.base_step(batch, batch_idx)
        self.validation_step_outputs_pred.append(pred)
        self.validation_step_outputs_lb.append(lb)
        self.log("val_loss", loss, prog_bar=True, batch_size=BATCH_SIZE)
        # return loss

    def on_validation_epoch_end(self):
        all_pred = torch.concat(self.validation_step_outputs_pred, dim=0)
        all_lb = torch.concat(self.validation_step_outputs_lb, dim=0)

        val_loss = self.loss_fn(all_pred.float(), all_lb.float())
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            # self.save_to_txt(all_pred)

        all_lb = torch.argmax(all_lb, dim=1)

        self.f1.update(all_pred, all_lb)
        self.mcc.update(all_pred, all_lb)
        self.average_precision.update(all_pred, all_lb)

        print(
            f"\nMetrics on validation set: "
            f"best_val_loss: {self.best_val_loss}, "
            f"f1: {self.f1.compute()}, "
            f"mcc: {self.mcc.compute()}, "
            f"average_precision: {self.average_precision.compute()} \n"
        )

        self.validation_step_outputs_lb.clear()
        self.validation_step_outputs_pred.clear()
        self.f1.reset()
        self.mcc.reset()
        self.average_precision.reset()

    def test_step(self, batch, batch_idx):
        _, lb, pred, loss, kingdom = self.base_step(batch, batch_idx)

        # Update outputs for calculate metrics on each class
        _pred = torch.argmax(pred, dim=1)
        _lb = torch.argmax(lb, dim=1)
        self.test_outputs_pred.extend(_pred.detach().cpu().numpy())
        self.test_outputs_lb.extend(_lb.detach().cpu().numpy())

        # Update outputs for calculate metrics on each kingdom
        for i, k in enumerate(kingdom):
            if k not in self.test_step_outputs.keys():
                self.test_step_outputs[k] = {}
                self.test_step_outputs[k]["pred"] = []
                self.test_step_outputs[k]["lb"] = []

            outputs = self.test_step_outputs[k]
            self.test_step_outputs[k]['pred'] = torch.stack([*outputs["pred"], pred[i]])
            self.test_step_outputs[k]['lb'] = torch.stack([*outputs["lb"], lb[i]])
            # outputs["pred"].append(pred[i])
            # outputs["lb"].append(lb[i])

        # self.test_step_outputs_lb.append(lb)
        # self.test_step_outputs_pred.append(pred)
        # lb = torch.argmax(lb, dim=1)

    def on_test_end(self) -> None:
        # Apply argmax on these outputs (only for label) and evaluate the metric results
        print(classification_report(self.test_outputs_lb, self.test_outputs_pred))

        #  all_pred = torch.concat(self.test_step_outputs_lb, dim=0)
        #  all_lb = torch.concat(self.test_step_outputs_lb, dim=0)
        #  all_lb = torch.argmax(all_lb, dim=1)
        #  all_test_outputs = {}

        # Calculate metrics on each kingdom
        f1_test = []
        mcc_test = []
        average_precision_test = []
        for key, _ in KINGDOM.items():
            all_pred = self.test_step_outputs[key]['pred']
            all_lb = torch.argmax(self.test_step_outputs[key]['lb'], dim=1)

            # Print the statistic (the following function has ERROR about syntax)
            self._save_results_to_txt(all_pred.detach().cpu(), all_lb.detach().cpu(), kingdom=key)

            self.f1.update(all_pred, all_lb)
            self.mcc.update(all_pred, all_lb)
            self.average_precision.update(all_pred, all_lb)
            f1_test.append(self.f1.compute().item())
            mcc_test.append(self.mcc.compute().item())
            average_precision_test.append(self.average_precision.compute().item())

            print(
                f'\nMetrics on test set of {key}: '
                f'f1: {self.f1.compute()}, '
                f'mcc: {self.mcc.compute()}, '
                f'average_precision: {self.average_precision.compute()} \n'
            )

            self.f1.reset()
            self.mcc.reset()
            self.average_precision.reset()

        metric_dict = {
            "kingdom": KINGDOM.keys(),
            "F1 Score": f1_test,
            "MCC Score": mcc_test,
            "Average Precision Score": average_precision_test
        }

        self._save_metrics_to_csv(metric_dict)

    @staticmethod
    def _save_results_to_txt(test_prediction_results, test_true_results, kingdom):
        if not os.path.exists(str(Path(WORKING_DIR, f"out/results"))):
            os.makedirs(f"out/results", exist_ok=True)

        softmax = Softmax()
        pred_path = f'out/results/{kingdom}_test_prediction_results_by_{MODEL}.txt'
        true_path = f'out/results/{kingdom}_test_true_results.txt'

        # print(test_prediction_results)

        np.savetxt(str(Path(WORKING_DIR, pred_path)), softmax(test_prediction_results), fmt="%.4f")
        np.savetxt(str(Path(WORKING_DIR, true_path)), test_true_results, fmt="%d")
        # if not os.path.exists(ut.abspath(true_path)):
        #     np.savetxt(ut.abspath(true_path), test_true_results, fmt="%d")

    @staticmethod
    def _save_metrics_to_csv(metric_dict):
        version = 0
        while os.path.exists(str(Path(WORKING_DIR, f'out/metrics/{MODEL}_metrics_results_{version}.csv'))):
            version += 1

        df = pd.DataFrame().from_dict(metric_dict)
        df.to_csv(str(Path(WORKING_DIR, f'out/metrics/{MODEL}_metrics_results_{version}.csv')), index_label="No")

In [ ]:
# access wandb
import wandb

wandb_api = '3c8685dbfce5b23f56fce47a675b7a3569dead2c'

wandb.login(key=wandb_api)

In [ ]:
# Training and Validation
import argparse
from typing import Union

import lightning as L
import torch
import wandb
# from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.loggers import WandbLogger


def parse_arguments():
    arg_parser = argparse.ArgumentParser()
    arg_parser.add_argument('--process', type=str, default='train', choices=['train', 'test', 'full'])
    arg_parser.add_argument('--env', type=str, default='local')
    arg_parser.add_argument('--log-dir', type=str, default='/logs')
    arg_parser.add_argument('--devices', type=Union[list[int], str, int], default='auto')
    # accelerator can be 'cpu', 'gpu', 'tpu' or use 'auto' instead of do not want to specify
    arg_parser.add_argument('--accelerator', type=str, default='auto')

    return arg_parser.parse_args()


if __name__ == '__main__':
    torch.set_float32_matmul_precision('medium')

    # CLI parsing arguments
    # args = parse_arguments()
    log_dir = str(Path(WORKING_DIR, LOG_DIR))
    # logger = TensorBoardLogger(save_dir=log_dir, name='tensorboard')
    logger = WandbLogger(save_dir=log_dir, name='wandb', project='SPPredictor')
    logger.experiment.config["batch_size"] = BATCH_SIZE

    sp_module = SPModule()
    sp_data_module = SPDataModule()
    trainer = L.Trainer(
        devices=DEVICES,
        accelerator=ACCELERATOR,
        max_epochs=EPOCHS,
        val_check_interval=1.0,
        logger=logger,
        # logger=False,
        # enable_checkpointing=False,
        # callbacks=[early_stopping, tqdm_progress_bar]
        callbacks=[model_checkpoint, early_stopping]
    )

    trainer.fit(sp_module, datamodule=sp_data_module)

    wandb.finish(quiet=True)


In [ ]:
# # Testing
# sp_data_module = SPDataModule(dataset_type=DATA)
# sp_module = SPModule.load_from_checkpoint(str(Path(BERT_CKPT) / f'bert_pretrained_epoch=1_kaggle.ckpt'))
# trainer = L.Trainer(
#     devices='auto',
#     accelerator='auto',
#     max_epochs=EPOCHS,
#     val_check_interval=1.0,
#     logger=False
# )

# trainer.test(sp_module, datamodule=sp_data_module)